# AlexNet on CIFAR-10

这个 Notebook 从数据集加载开始，完整展示 `CIFAR-10 -> Resize(224x224) -> AlexNet` 的训练与分析流程。

内容包括：
- 数据集下载、预处理与可视化
- `DataLoader` 构建
- 适用于 `224x224` 输入的 AlexNet 实现
- 模型逐层解读与特征图尺寸分析
- 训练、验证与预测展示
- 参数量统计与 AlexNet 设计逻辑说明

## 1. 环境准备

这里使用 `PyTorch` 和 `torchvision`。如果本地缺少依赖，需要先自行安装：

```bash
pip install torch torchvision matplotlib
```

In [ ]:
# 数学工具主要用于后面计算子图排布等辅助逻辑
import math
# dataclass 用于集中管理实验配置，避免超参数散落在各处
from dataclasses import dataclass

# matplotlib 用于样本、训练曲线和预测结果可视化
import matplotlib.pyplot as plt
# PyTorch 核心库
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
# torchvision 提供常用视觉数据集与图像变换工具
from torchvision import datasets, transforms

# 设置绘图风格，便于 notebook 展示
plt.style.use('seaborn-v0_8')
# 固定随机种子，便于复现结果
torch.manual_seed(42)

# 优先使用 GPU；如果本机没有 CUDA，则退回 CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    # 数据集下载和缓存目录
    data_root: str = './data'
    # 将 CIFAR-10 统一拉伸到 224x224，以适配 AlexNet 输入风格
    image_size: int = 224
    # 每个 batch 的样本数
    batch_size: int = 64
    # DataLoader 的并行加载进程数
    num_workers: int = 2
    # Adam 优化器学习率
    lr: float = 1e-3
    # 默认训练轮数；可按机器性能自行调整
    epochs: int = 5


cfg = Config()
cfg

## 2. 加载 CIFAR-10 并拉伸到 224x224

原始 `CIFAR-10` 图片大小是 `32x32`，而经典 AlexNet 面向的大尺寸输入更接近 `224x224`。这里先做尺寸拉伸，再进行标准化。

这样做的好处是：
- 可以更接近原始 AlexNet 的输入设定
- 卷积核、步幅和池化层的尺寸变化更容易讲清楚
- 更适合做模型结构解读

代价是：
- 图片被放大后不会新增真实细节
- 训练成本会明显高于直接使用 `32x32` 输入

In [ ]:
# 这里使用 CIFAR-10 常见的均值和标准差做归一化
cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std = (0.2470, 0.2435, 0.2616)

# 训练集变换：先拉伸到 224x224，再做随机翻转增强，最后转 tensor 并归一化
train_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

# 测试集不做随机增强，只保留尺寸调整和归一化
test_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

# download=True 表示本地没有数据时自动下载
train_dataset = datasets.CIFAR10(
    root=cfg.data_root,
    train=True,
    download=True,
    transform=train_transform,
)

test_dataset = datasets.CIFAR10(
    root=cfg.data_root,
    train=False,
    download=True,
    transform=test_transform,
)

# CIFAR-10 的类别名称列表
classes = train_dataset.classes
classes

In [ ]:
def denormalize(image_tensor, mean, std):
    # 反归一化，便于把图像恢复到可视化时更自然的像素范围
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return image_tensor * std + mean


# 展示若干个被拉伸到 224x224 的样本，直观看看输入长什么样
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, idx in zip(axes.flatten(), range(8)):
    image, label = train_dataset[idx]
    image = denormalize(image, cifar10_mean, cifar10_std).permute(1, 2, 0).clamp(0, 1)
    ax.imshow(image)
    ax.set_title(classes[label])
    ax.axis('off')

plt.suptitle('CIFAR-10 samples resized to 224x224', fontsize=16)
plt.tight_layout()
plt.show()

## 3. 构建 DataLoader

In [ ]:
# 训练集需要打乱顺序，减少模型对样本顺序的依赖
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    # 如果使用 GPU，pin_memory 往往能让数据搬运更高效
    pin_memory=torch.cuda.is_available(),
)

# 测试集不需要打乱，便于稳定评估
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

# 查看一个 batch 的形状，确认数据维度是否符合预期
images, labels = next(iter(train_loader))
print('batch image shape:', images.shape)
print('batch label shape:', labels.shape)

## 4. AlexNet 实现

下面的结构基本沿用经典 AlexNet 的主干思路：
- 前部使用大卷积核和较大步幅，快速压缩空间尺寸
- 中间堆叠多个 `3x3` 卷积，增强表征能力
- 末端使用全连接层完成分类

这里将最后输出类别数改为 `10`，以适配 `CIFAR-10`。

In [ ]:
class AlexNetCIFAR10(nn.Module):
    def __init__(self, num_classes=10, dropout=0.5):
        super().__init__()
        # features 部分负责逐层提取视觉特征
        self.features = nn.Sequential(
            # 第一层大卷积核 + 大步幅：快速扩大感受野并压缩空间尺寸
            nn.Conv2d(3, 64, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),

            # 第二层增加通道数，提取更丰富的中层模式
            nn.Conv2d(64, 192, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),

            # 接下来的三个 3x3 卷积块是经典 AlexNet 的核心表征阶段
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        # 将卷积输出统一整理成 6x6，便于接入经典全连接头
        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))
        # classifier 部分负责把高维特征映射成最终类别
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        # 卷积特征提取
        x = self.features(x)
        # 统一空间尺寸
        x = self.avgpool(x)
        # 展平成二维张量，以便输入全连接层
        x = torch.flatten(x, 1)
        # 输出每个类别的 logits
        x = self.classifier(x)
        return x


model = AlexNetCIFAR10().to(device)
model

## 5. 逐层尺寸变化分析

AlexNet 的一个关键学习点是：空间尺寸如何逐层缩小，通道数如何逐层增加。下面通过一次前向传播跟踪每一层输出张量的尺寸。

In [ ]:
def inspect_feature_shapes(model, input_shape=(1, 3, 224, 224)):
    # 构造一个假的输入张量，只用于追踪网络内部尺寸变化
    x = torch.randn(input_shape)
    print(f'input: {tuple(x.shape)}')

    for idx, layer in enumerate(model.features):
        # 每经过一层，就打印一次输出 shape，便于分析空间尺寸和通道数变化
        x = layer(x)
        print(f'features[{idx}] {layer.__class__.__name__:<12} -> {tuple(x.shape)}')

    x = model.avgpool(x)
    print(f'avgpool           AdaptiveAvgPool2d -> {tuple(x.shape)}')

    x = torch.flatten(x, 1)
    print(f'flatten                         -> {tuple(x.shape)}')

    for idx, layer in enumerate(model.classifier):
        x = layer(x)
        print(f'classifier[{idx}] {layer.__class__.__name__:<12} -> {tuple(x.shape)}')


inspect_feature_shapes(model.cpu())
model = model.to(device)

### 结构解读

1. 第一层 `11x11, stride=4` 卷积
   - 作用是快速扩大感受野，同时明显压缩空间尺寸。
   - 对 `224x224` 输入来说，这一步会把图片从高分辨率表征切到更粗粒度特征。

2. 第一、第二次池化
   - 进一步降低计算量。
   - 让后面的卷积层更多关注高级模式，而不是局部像素波动。

3. 中间三个 `3x3` 卷积块
   - 这是 AlexNet 表征能力的重要来源。
   - 小卷积核堆叠可以在控制参数量的同时增加非线性表达。

4. `AdaptiveAvgPool2d((6, 6))`
   - 作用是把卷积输出稳定整理到全连接层可接受的形状。
   - 这样结构解释更清楚，也保留了经典 AlexNet 的全连接头部设计。

5. 三层全连接分类器
   - 前两层大宽度线性层负责融合全局语义。
   - 最后一层将特征映射到 `10` 个类别。

## 6. 参数量统计

In [ ]:
def count_parameters(model):
    # 只统计需要训练的参数
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


total_params = count_parameters(model)
print(f'Trainable parameters: {total_params:,}')

从这里可以看到，AlexNet 的大部分参数集中在后面的全连接层，这也是经典 CNN 在参数效率上的一个典型特点。与很多现代网络相比，它的卷积主干不算特别深，但分类头参数非常大。

## 7. 训练与验证函数

In [ ]:
# 多分类任务常用交叉熵损失
criterion = nn.CrossEntropyLoss()
# 这里使用 Adam，收敛通常比纯 SGD 更省调参成本
optimizer = optim.Adam(model.parameters(), lr=cfg.lr)


def train_one_epoch(model, dataloader, criterion, optimizer, device):
    # 训练模式会启用 Dropout 等训练时行为
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in dataloader:
        # 把一个 batch 的数据搬到目标设备上
        images = images.to(device)
        labels = labels.to(device)

        # 清空上一轮梯度，避免梯度累积
        optimizer.zero_grad()
        # 前向传播：得到每个类别的预测分数
        outputs = model(images)
        # 计算预测与真实标签之间的损失
        loss = criterion(outputs, labels)
        # 反向传播：计算梯度
        loss.backward()
        # 参数更新
        optimizer.step()

        # 按样本数累加损失，便于后面计算 epoch 平均值
        running_loss += loss.item() * images.size(0)
        # 取 logits 最大值对应的类别作为预测结果
        preds = outputs.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, running_correct / total


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    # 评估模式会关闭 Dropout 等训练期随机行为
    model.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in dataloader:
        # 评估阶段同样需要把数据送到对应设备
        images = images.to(device)
        labels = labels.to(device)

        # 在 no_grad 环境下只做前向传播，不计算梯度
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, running_correct / total

## 8. 训练主循环

下面代码默认训练 `5` 个 epoch。你可以根据机器性能自行调整。由于 `224x224` 输入比原始 `32x32` 成本高很多，如果在 CPU 上运行，训练会比较慢。

In [ ]:
# 用 history 记录每个 epoch 的训练和验证指标，方便后面画曲线
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
}

# 训练主循环：每轮先训练，再在测试集上评估
for epoch in range(cfg.epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)

    # 把本轮结果写入 history，后面统一可视化
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(
        f'Epoch [{epoch + 1}/{cfg.epochs}] '
        f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} '
        f'val_loss={val_loss:.4f} val_acc={val_acc:.4f}'
    )

In [ ]:
# 横轴使用 epoch 序号，纵轴分别展示损失和准确率变化
epochs = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, history['train_loss'], label='train loss')
axes[0].plot(epochs, history['val_loss'], label='val loss')
axes[0].set_title('Loss curves')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs, history['train_acc'], label='train acc')
axes[1].plot(epochs, history['val_acc'], label='val acc')
axes[1].set_title('Accuracy curves')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. 预测结果展示

In [ ]:
@torch.no_grad()
def show_predictions(model, dataloader, class_names, device, num_images=8):
    # 切换到评估模式，保证推理行为稳定
    model.eval()
    # 取一个 batch 做演示
    images, labels = next(iter(dataloader))
    images = images.to(device)
    labels = labels.to(device)

    # 前向传播并取预测类别
    logits = model(images)
    preds = logits.argmax(dim=1)

    # 按两行排布可视化结果
    fig, axes = plt.subplots(2, math.ceil(num_images / 2), figsize=(16, 6))
    axes = axes.flatten()

    for i in range(num_images):
        # 为了可视化，需要把归一化后的 tensor 还原到接近原始像素范围
        image = denormalize(images[i].cpu(), cifar10_mean, cifar10_std).permute(1, 2, 0).clamp(0, 1)
        axes[i].imshow(image)
        axes[i].set_title(f'true: {class_names[labels[i]]}\npred: {class_names[preds[i]]}')
        axes[i].axis('off')

    for i in range(num_images, len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()


show_predictions(model, test_loader, classes, device)

## 10. 为什么这里要把 CIFAR-10 拉伸到 224x224？

### 原因 1：更贴近 AlexNet 的经典输入设定
AlexNet 最初面向的是更大尺寸自然图像。直接保留 `224x224` 输入逻辑，可以让第一层大卷积核和后续池化的作用更容易理解。

### 原因 2：便于解释特征图压缩过程
如果直接用 `32x32`，原始 AlexNet 的若干层配置会压缩得过快，结构解释会变得不自然。拉伸后，尺寸变化路径更接近经典 CNN 教材中的讲解方式。

### 原因 3：适合教学和结构分析
这个 Notebook 的重点不仅是训练一个分类器，更是解释：
- 大卷积核为什么有效
- 池化为什么能压缩空间维度
- 通道数为什么逐层增加
- 全连接层为什么带来大量参数

### 需要注意的限制
把 `32x32` 图片放大到 `224x224` 并不会凭空创造新纹理，因此这个流程更适合作为结构学习和模型分析示例，而不一定是 `CIFAR-10` 上最节省算力的实用方案。

## 11. 可继续扩展的方向

- 增加混淆矩阵分析不同类别的误判模式
- 可视化中间层特征图
- 对比 `32x32` 版本 AlexNet 与 `224x224` 版本的性能和开销
- 对比 AlexNet、VGG、ResNet 的结构差异